# Visual Product Search Engine — Evaluation (Config A and B)

This notebook evaluates retrieval performance for Config A and Config B.

---

### Metrics computed
| Metric | Meaning |
|--------|---------|
| Recall@K | Fraction of queries with at least 1 correct result in top-K |
| NDCG@K | Position-aware metric — correct results ranked higher = better score |
| mAP@K | Mean average precision up to K — rewards finding all correct items early |

K values: 5, 10, 15

---

### How evaluation works
1. For each query image, encode it with CLIP (image only — no caption for query)
2. Search the gallery FAISS index → top-K results
3. Check which results share the same item_id as the query → ground truth match
4. Compute metrics over all queries
5. Report mean ± std over multiple seeds (your roll numbers)

---

### Inputs
- `yolo-bbox-crops-v1` — query crops + master_crops.csv
- `clip-indexes-ab` — FAISS indexes + gallery_metadata.csv + embeddings
- `blipcaptionsoutput` — NOT needed for evaluation (query images have no captions)

## Step 1: Install and Import

In [1]:
!pip uninstall -y faiss faiss-gpu
!pip install ftfy regex transformers faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 77.8 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch
import faiss
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print('Libraries imported!')

Device : cuda
Libraries imported!


## Step 2: Paths and Seeds

**IMPORTANT: Set SEEDS to your team's roll numbers before running.**

In [3]:
# ============================================================
#  SET YOUR TEAM ROLL NUMBERS AS SEEDS
# ============================================================
SEEDS = [550, 537, 585, 35]   # team roll numbers
# ============================================================

CROPS_DATASET  = '/kaggle/input/datasets/harshitabansal307/yolo-bbox-crops-v1'
INDEXES_DATASET = '/kaggle/input/datasets/likithareddy2508/clip-indexes-ab'

K_VALUES   = [5, 10, 15]
BATCH_SIZE = 64
CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'

# Configs to evaluate
# Each entry: (config_name, index_filename, alpha)
CONFIGS = [
    ('Config_A_alpha1.0',  'faiss_index_A_alpha10.bin', 1.0),
    ('Config_B_alpha0.7',  'faiss_index_B_alpha07.bin', 0.7),
    ('Config_B_alpha0.5',  'faiss_index_B_alpha05.bin', 0.5),
]

for label, path in [('CROPS_DATASET', CROPS_DATASET), ('INDEXES_DATASET', INDEXES_DATASET)]:
    status = 'Found ✓' if os.path.exists(path) else 'NOT FOUND ✗'
    print(f'[{status}] {label}')

print(f'\nSeeds : {SEEDS}')
print(f'K values : {K_VALUES}')

[Found ✓] CROPS_DATASET
[Found ✓] INDEXES_DATASET

Seeds : [550, 537, 585, 35]
K values : [5, 10, 15]


## Step 3: Load Query Data and Gallery Metadata

In [4]:
master_df = pd.read_csv(os.path.join(CROPS_DATASET, 'master_crops.csv'))
query_df  = master_df[master_df['split'] == 'query'].reset_index(drop=True)

# Remap query crop paths
def remap_path(old_path):
    if pd.isna(old_path):
        return old_path
    relative = old_path.replace('/kaggle/working/', '').replace('/kaggle/input/', '')
    for ds in ['yolocroppeddataset/', 'yolo-bbox-crops-v1/', 'datasets/harshitabansal307/yolo-bbox-crops-v1/']:
        relative = relative.replace(ds, '')
    return os.path.join(CROPS_DATASET, relative)

query_df['crop_path_new'] = query_df['crop_path'].apply(remap_path)
query_df['crop_exists']   = query_df['crop_path_new'].apply(
    lambda p: os.path.exists(p) if isinstance(p, str) else False
)

if query_df['crop_exists'].sum() < len(query_df) * 0.9:
    def direct_path(image_name):
        relative = image_name[4:] if image_name.startswith('img/') else image_name
        for subdir in ['data/bbox_crops', 'data/yolo_crops']:
            p = os.path.join(CROPS_DATASET, subdir, relative)
            if os.path.exists(p):
                return p
        return os.path.join(CROPS_DATASET, 'data/bbox_crops', relative)
    query_df['crop_path_new'] = query_df['image_name'].apply(direct_path)
    query_df['crop_exists']   = query_df['crop_path_new'].apply(os.path.exists)

valid_query_df = query_df[query_df['crop_exists']].reset_index(drop=True)

# Load gallery metadata
gallery_meta = pd.read_csv(os.path.join(INDEXES_DATASET, 'gallery_metadata.csv'))

print(f'Query images  : {len(valid_query_df):,}')
print(f'Gallery items : {len(gallery_meta):,}')
print(f'Unique query item_ids : {valid_query_df["item_id"].nunique():,}')

Query images  : 14,218
Gallery items : 12,612
Unique query item_ids : 3,985


## Step 4: Load CLIP and Encode All Query Images

In [5]:
print('Loading CLIP...')
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
for param in clip_model.parameters():
    param.requires_grad = False
clip_model.eval()

EMBED_DIM = clip_model.config.projection_dim
print(f'CLIP loaded! Embedding dim: {EMBED_DIM}')

# Encode all query images — this is the SAME for all configs
# Query always uses image-only encoding (no caption for query image)
n_query = len(valid_query_df)
query_image_embs = np.zeros((n_query, EMBED_DIM), dtype=np.float32)

print(f'Encoding {n_query:,} query images...')
for batch_start in tqdm(range(0, n_query, BATCH_SIZE), desc='Query encoding'):
    batch = valid_query_df.iloc[batch_start : batch_start + BATCH_SIZE]
    images = []
    valid_idx = []
    for i, (_, row) in enumerate(batch.iterrows()):
        try:
            img = Image.open(row['crop_path_new']).convert('RGB')
            images.append(img)
            valid_idx.append(i)
        except:
            pass
    if not images:
        continue
    inputs = clip_processor(images=images, return_tensors='pt', padding=True).to(DEVICE)
    with torch.no_grad():
        out = clip_model.get_image_features(**inputs)
        feats = out.pooler_output if hasattr(out, 'pooler_output') and not isinstance(out, torch.Tensor) else out
    feats = feats / feats.norm(dim=-1, keepdim=True)
    for local_i, global_i in enumerate(valid_idx):
        query_image_embs[batch_start + global_i] = feats[local_i].cpu().numpy()

print(f'Query embeddings shape: {query_image_embs.shape}')

Loading CLIP...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded! Embedding dim: 512
Encoding 14,218 query images...



Query encoding: 100%|██████████| 223/223 [02:10<00:00,  1.71it/s]

Query embeddings shape: (14218, 512)


## Step 5: Define Metric Functions

In [6]:
# Metric functions with correct denominators
# Key fix: use n_relevant_in_gallery (actual count of gallery matches)
# as denominator for NDCG and mAP, NOT len(relevant_ids) which was always 1

print('Metric functions defined ✓')

# Sanity check
test_retrieved = ['A', 'B', 'A', 'C', 'A']
test_item_id   = 'A'
test_n_rel     = 3  # 3 gallery images have item_id A
# Recall@5: A appears → 1
assert int(any(r == test_item_id for r in test_retrieved[:5])) == 1
print(f'Test Recall@5  : 1  ✓')
# NDCG@5: hits at ranks 1,3,5; ideal hits at ranks 1,2,3
dcg  = 1/np.log2(2) + 1/np.log2(4) + 1/np.log2(6)
idcg = 1/np.log2(2) + 1/np.log2(3) + 1/np.log2(4)
print(f'Test NDCG@5    : {dcg/idcg:.3f}  ✓  (bounded 0-1)')
# mAP@5: prec_sum = 1/1 + 2/3 + 3/5 = 2.267; denom = min(3,5)=3 → 0.756
prec_sum = 1/1 + 2/3 + 3/5
print(f'Test mAP@5     : {prec_sum/3:.3f}  ✓  (bounded 0-1)')


Metric functions defined ✓
Test Recall@5  : 1  ✓
Test NDCG@5    : 0.885  ✓  (bounded 0-1)
Test mAP@5     : 0.756  ✓  (bounded 0-1)


## Step 6: Run Evaluation for Each Config and Each Seed

In [7]:
gallery_item_ids = gallery_meta['item_id'].tolist()

# Pre-build lookup: item_id -> count of gallery images with that item_id
# This is the correct denominator for NDCG and mAP
gallery_item_counts = gallery_meta['item_id'].value_counts().to_dict()

all_results = []

for config_name, index_file, alpha in CONFIGS:
    print(f'\n=== Evaluating {config_name} ===')

    index_path = os.path.join(INDEXES_DATASET, index_file)
    index = faiss.read_index(index_path)
    print(f'  Index loaded: {index.ntotal:,} vectors')

    search_embs = query_image_embs.astype(np.float32)
    seed_scores = {k: {'recall': [], 'ndcg': [], 'map': []} for k in K_VALUES}

    for seed in SEEDS:
        np.random.seed(seed)
        torch.manual_seed(seed)

        n_sample = max(500, int(0.2 * len(valid_query_df)))
        n_sample = min(n_sample, len(valid_query_df))
        sample_indices = np.random.choice(len(valid_query_df), n_sample, replace=False)

        sample_embs     = search_embs[sample_indices]
        sample_item_ids = valid_query_df.iloc[sample_indices]['item_id'].tolist()

        distances, indices = index.search(sample_embs, 15 + 1)

        per_k_recall = {k: [] for k in K_VALUES}
        per_k_ndcg   = {k: [] for k in K_VALUES}
        per_k_map    = {k: [] for k in K_VALUES}

        for q_item_id, result_indices in zip(sample_item_ids, indices):
            retrieved_item_ids = [gallery_item_ids[i] for i in result_indices if i < len(gallery_item_ids)]

            # n_relevant: actual number of gallery images for this item_id
            n_relevant = gallery_item_counts.get(q_item_id, 1)

            for k in K_VALUES:
                retrieved_k = retrieved_item_ids[:k]

                # Recall@K: 1 if any match in top-k
                per_k_recall[k].append(int(any(r == q_item_id for r in retrieved_k)))

                # NDCG@K
                dcg = sum(
                    1.0 / np.log2(rank + 2)
                    for rank, r in enumerate(retrieved_k)
                    if r == q_item_id
                )
                n_ideal = min(n_relevant, k)
                idcg = sum(1.0 / np.log2(i + 2) for i in range(n_ideal))
                per_k_ndcg[k].append(dcg / idcg if idcg > 0 else 0.0)

                # mAP@K
                hits, prec_sum = 0, 0.0
                for rank, r in enumerate(retrieved_k, start=1):
                    if r == q_item_id:
                        hits += 1
                        prec_sum += hits / rank
                denom = min(n_relevant, k)
                per_k_map[k].append(prec_sum / denom if denom > 0 else 0.0)

        for k in K_VALUES:
            seed_scores[k]['recall'].append(np.mean(per_k_recall[k]))
            seed_scores[k]['ndcg'].append(np.mean(per_k_ndcg[k]))
            seed_scores[k]['map'].append(np.mean(per_k_map[k]))

        print(f'  Seed {seed}: Recall@10={np.mean(per_k_recall[10]):.4f}  NDCG@10={np.mean(per_k_ndcg[10]):.4f}  mAP@10={np.mean(per_k_map[10]):.4f}')

    for k in K_VALUES:
        all_results.append({
            'config': config_name, 'K': k,
            'Recall_mean': np.mean(seed_scores[k]['recall']),
            'Recall_std':  np.std(seed_scores[k]['recall']),
            'NDCG_mean':   np.mean(seed_scores[k]['ndcg']),
            'NDCG_std':    np.std(seed_scores[k]['ndcg']),
            'mAP_mean':    np.mean(seed_scores[k]['map']),
            'mAP_std':     np.std(seed_scores[k]['map']),
        })

print('\nEvaluation complete!')



=== Evaluating Config_A_alpha1.0 ===
  Index loaded: 12,612 vectors
  Seed 550: Recall@10=0.5652  NDCG@10=0.2420  mAP@10=0.1700
  Seed 537: Recall@10=0.5519  NDCG@10=0.2274  mAP@10=0.1558
  Seed 585: Recall@10=0.5853  NDCG@10=0.2408  mAP@10=0.1642
  Seed 35: Recall@10=0.5558  NDCG@10=0.2384  mAP@10=0.1672

=== Evaluating Config_B_alpha0.7 ===
  Index loaded: 12,612 vectors
  Seed 550: Recall@10=0.5596  NDCG@10=0.2396  mAP@10=0.1696
  Seed 537: Recall@10=0.5480  NDCG@10=0.2256  mAP@10=0.1554
  Seed 585: Recall@10=0.5779  NDCG@10=0.2396  mAP@10=0.1643
  Seed 35: Recall@10=0.5667  NDCG@10=0.2382  mAP@10=0.1664

=== Evaluating Config_B_alpha0.5 ===
  Index loaded: 12,612 vectors
  Seed 550: Recall@10=0.5192  NDCG@10=0.2108  mAP@10=0.1462
  Seed 537: Recall@10=0.5100  NDCG@10=0.1983  mAP@10=0.1338
  Seed 585: Recall@10=0.5375  NDCG@10=0.2076  mAP@10=0.1383
  Seed 35: Recall@10=0.5195  NDCG@10=0.2060  mAP@10=0.1406

Evaluation complete!


## Step 7: Print Results Table

In [8]:
results_df = pd.DataFrame(all_results)

print('\n=== ABLATION STUDY RESULTS ===')
print(f'Seeds used: {SEEDS}')
print(f'Format: mean ± std')
print()

for config_name, _, _ in CONFIGS:
    config_rows = results_df[results_df['config'] == config_name]
    print(f'Config: {config_name}')
    print(f'  {"K":>4}  {"Recall@K":>12}  {"NDCG@K":>12}  {"mAP@K":>12}')
    print(f'  {"─"*50}')
    for _, row in config_rows.iterrows():
        print(
            f'  K={int(row["K"]):>2}  '
            f'{row["Recall_mean"]:.4f}±{row["Recall_std"]:.4f}  '
            f'{row["NDCG_mean"]:.4f}±{row["NDCG_std"]:.4f}  '
            f'{row["mAP_mean"]:.4f}±{row["mAP_std"]:.4f}'
        )
    print()

# Save results
results_df.to_csv('/kaggle/working/evaluation_results_AB.csv', index=False)
print('Results saved to evaluation_results_AB.csv')


=== ABLATION STUDY RESULTS ===
Seeds used: [550, 537, 585, 35]
Format: mean ± std

Config: Config_A_alpha1.0
     K      Recall@K        NDCG@K         mAP@K
  ──────────────────────────────────────────────────
  K= 5  0.4960±0.0094  0.2342±0.0056  0.1724±0.0051
  K=10  0.5645±0.0129  0.2371±0.0058  0.1643±0.0053
  K=15  0.6012±0.0127  0.2441±0.0059  0.1649±0.0054

Config: Config_B_alpha0.7
     K      Recall@K        NDCG@K         mAP@K
  ──────────────────────────────────────────────────
  K= 5  0.4828±0.0112  0.2289±0.0056  0.1695±0.0049
  K=10  0.5630±0.0109  0.2358±0.0059  0.1639±0.0053
  K=15  0.6070±0.0100  0.2436±0.0061  0.1649±0.0054

Config: Config_B_alpha0.5
     K      Recall@K        NDCG@K         mAP@K
  ──────────────────────────────────────────────────
  K= 5  0.4371±0.0079  0.1972±0.0041  0.1432±0.0041
  K=10  0.5215±0.0099  0.2057±0.0046  0.1397±0.0045
  K=15  0.5704±0.0119  0.2149±0.0053  0.1414±0.0047

Results saved to evaluation_results_AB.csv
